# core

> AoC token cache, puzzle operations (`input_data`, `examples`, `title`, `answer`, `submit_answer`), and `Puzzle` interface.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.all import *
from baocd.utils import test_beq

In [ ]:
#| export
import webbrowser
from functools import cached_property
from types import SimpleNamespace
from basedpl import *
from baocd.utils import (
    AocdError, DeadTokenError, PuzzleLockedError,
    aiter, atomic_write_file, beq, bhas, bjoin, brstrip_nl, bsplit_lines,
    bstartswith, bstr, bstrip, bwords, coerce, env_get, expand_path,
    http_get, http_post, is_file, join_path, pystr, read_json, read_text,
    rec_has, rec_set, tag_texts, unescape_html, write_json, write_text,
)

## Cache Directories & Tacit Helpers

In [ ]:
#| export
AOCD_DATA_DIR = expand_path(env_get("AOCD_DIR", "~/.config/aocd"))
AOCD_CONFIG_DIR = expand_path(env_get("AOCD_CONFIG_DIR", AOCD_DATA_DIR))
EST_OFFSET = -5 * 3600
NON_ANSWER = ("", b"", None, b"None", "None")
_token2id = None

_norm_part = atop(
    -1 | c,
    replace | catenate(enclose(Array("2")), enclose(Array("b"))),
    replace | catenate(enclose(Array("1")), enclose(Array("a"))),
)
_wait_groups = (r | r"You have (?:(\d+)m )?(\d+)s left to wait")["groups"]
_norm_social = atop(
    replace | Array(["https://www.reddit.com/u/", "reddit/"]),
    replace | Array(["https://github.com/", "github/"]),
    replace | Array(["https://twitter.com/", "twitter/"]),
    (Array("Link to ") | catenate).undo,
)
_now_iso = atop(dict(pattern="%Y-%m-%d %H:%M:%S-05:00", zone=EST_OFFSET) | date.undo, time)


def _is_non_answer(val):
    if isinstance(val, Array):
        return (beq(val, Array("")) or beq(val, Array("None"))) if val._inner.text is not None else False
    return val in NON_ANSWER


def is_final_day(year, day): return (int(day) == 12) if int(year) >= 2025 else (int(day) == 25)

## Token Discovery & Shared `aocd` Cache Directory

In [ ]:
#| export
def default_token():
    "Read session token from `AOC_SESSION` or `~/.config/aocd/token`."
    if (ck := env_get("AOC_SESSION")) is not None and len(bstrip(ck)): return bstrip(ck)
    tok_f = join_path(AOCD_CONFIG_DIR, "token")
    try:
        if len(ws := bwords(read_text(tok_f))): return ws[0]
    except FileNotFoundError: pass
    raise AocdError(f"Missing AoC session token: set AOC_SESSION or save it to {pystr(tok_f)}")


def get_owner(token):
    "Find owner of an AoC session token (`'authtype.username.userid'`) via pure-BPL `•r`."
    tok = bstr(token)
    tail = pystr(take(-4, tok))
    resp = http_get("https://adventofcode.com/settings", token=tok)
    if int(resp.status) != 200 or bool(bhas("[Log In]", resp.body)): raise DeadTokenError(f"the auth token ...{tail} is dead")
    codes = tag_texts(resp.body, "code")
    if not codes or not bool(bhas("-", codes[0])): raise DeadTokenError(f"the auth token ...{tail} is dead")
    id_m = r(codes[0], r"-([^\s-]+)")
    if not len(id_m.groups): raise DeadTokenError(f"the auth token ...{tail} is dead")
    userid = id_m.groups[0][0]

    auth_src = username = Array("unknown")
    link_m = r(resp.body, r"Link to ([^\s<]+)")
    goog_m = r(resp.body, r"googleusercontent\.com[^>]*>([^<]+)")
    if len(link_m.text):
        parts = r(_norm_social(link_m.text[0]), r"^([^/]+)/(.+)$")
        if len(parts.groups): auth_src, username = parts.groups[0][0], parts.groups[0][1]
    elif len(goog_m.groups):
        auth_src, username = Array("google"), unescape_html(goog_m.groups[0][0])
    return bjoin(".", Array([auth_src, username, userid]))


def user_id(token):
    "Cached `'authtype.username.userid'` stored in `token2id.json` (shared with `aocd`)."
    global _token2id
    tok = bstr(token)
    p = join_path(AOCD_CONFIG_DIR, "token2id.json")
    if _token2id is None:
        try: _token2id = read_json(p)
        except FileNotFoundError: _token2id = json("{}")
    k = pystr(tok)
    if not rec_has(_token2id, k):
        owner = get_owner(tok)
        _token2id = rec_set(_token2id, k, owner)
        write_json(p, _token2id)
        return owner
    return _token2id[k]


def puzzle_url(year, day): return Array(f"https://adventofcode.com/{int(year)}/day/{int(day)}")


def puzzle_paths(year, day, token):
    "Cache file paths for `(year, day, token)` matching `~/.config/aocd` layout."
    stem, mdir = f"{int(year)}_{int(day):02d}", join_path(AOCD_DATA_DIR, user_id(token))
    return Array(
        input_data=join_path(mdir, f"{stem}_input.txt"),
        answer_a=join_path(mdir, f"{stem}a_answer.txt"),
        answer_b=join_path(mdir, f"{stem}b_answer.txt"),
        submit_results=join_path(mdir, f"{stem}_post.json"),
        prose0=join_path(AOCD_DATA_DIR, "prose", f"{stem}_prose.0.html"),
        prose1=join_path(mdir, f"{stem}_prose.1.html"),
        prose2=join_path(mdir, f"{stem}_prose.2.html"),
    )

## Functional Puzzle Operations (Input, Prose, Examples, Answers, & Guardrailed Submit)

In [ ]:
#| export
def raw_input_data(year, day, token):
    "Raw (unsplit) puzzle input text as a BasedPL `Array` string (cached on disk)."
    p = puzzle_paths(year, day, token).input_data
    try: return brstrip_nl(read_text(p))
    except FileNotFoundError: pass
    url = catenate(puzzle_url(year, day), Array("/input"))
    resp = http_get(url, token=token)
    st = int(resp.status)
    if st >= 400:
        if st == 404: raise PuzzleLockedError(f"{year}/{day:02d} not available yet")
        raise AocdError(f"HTTP {st} at {pystr(url)}")
    atomic_write_file(p, resp.body)
    return brstrip_nl(resp.body)


def _save_answer(year, day, value, part, token):
    p, txt = puzzle_paths(year, day, token)[f"answer_{part}"], bstrip(bstr(value))
    if is_file(p) and beq(txt, read_text(p)): return
    write_text(p, txt)


def _request_prose(year, day, token):
    url = puzzle_url(year, day)
    resp = http_get(url, token=token)
    if int(resp.status) != 200: raise AocdError(f"HTTP {int(resp.status)} at {pystr(url)}")
    text, paths = resp.body, puzzle_paths(year, day, token)
    ans_m = r(text, r"Your puzzle answer was <code>([^<]+)</code>").groups
    if bool(bhas("Both parts of this puzzle are complete!", text)):
        if not is_file(paths.prose2): write_text(paths.prose2, text)
        if len(ans_m) >= 1: _save_answer(year, day, unescape_html(ans_m[0][0]), "a", token)
        if len(ans_m) >= 2 and not is_final_day(year, day): _save_answer(year, day, unescape_html(ans_m[1][0]), "b", token)
    elif bool(bhas("The first half of this puzzle is complete!", text)):
        if not is_file(paths.prose1): write_text(paths.prose1, text)
        if len(ans_m) >= 1: _save_answer(year, day, unescape_html(ans_m[0][0]), "a", token)
    elif not is_file(paths.prose0) and bool(bhas("Advent of Code", text)):
        write_text(paths.prose0, text)
    return text


def puzzle_prose(year, day, token):
    "Read cached puzzle HTML prose or fetch from adventofcode.com."
    paths = puzzle_paths(year, day, token)
    for pth in (paths.prose2, paths.prose1, paths.prose0):
        if is_file(pth): return read_text(pth)
    return _request_prose(year, day, token)


def puzzle_title(year, day, token):
    "Puzzle title as a BasedPL `Array` string, stripped via inverse catenate (`,⁻`)."
    h2s = tag_texts(puzzle_prose(year, day, token), "h2", strip_ws=True)
    if not h2s: raise AocdError("heading not found")
    try: return (catenate | Array(" ---")).undo(catenate.undo(Array(f"--- Day {int(day)}: "), h2s[0]))
    except BplError: raise AocdError(f"unexpected h2 text: {pystr(h2s[0])}")


def puzzle_examples(year, day, token):
    "Extract puzzle example(s) via pure-BPL `•r` with pre-split BasedPL line `Array` `input_data`."
    arts = r(puzzle_prose(year, day, token), r"(?is)<article\b[^>]*>(.*?)</article>").groups
    if not len(arts): return []
    art_a = arts[0][0]
    pres = tag_texts(art_a, "pre")
    if not pres: return []
    codes_a = tag_texts(art_a, "code")
    ans_a = (ws[-1] if len(ws := bwords(codes_a[-1])) else None) if codes_a else None
    ans_b = None
    if len(arts) >= 2 and not is_final_day(year, day):
        codes_b = tag_texts(arts[1][0], "code")
        if codes_b and len(ws := bwords(codes_b[-1])): ans_b = ws[-1]
    return [SimpleNamespace(input_data=bsplit_lines(pres[0]), answer_a=ans_a, answer_b=ans_b, answers=(ans_a, ans_b))]


def puzzle_answer(year, day, part, token):
    "Return cached/fetched answer for `(year, day, part)` as a BasedPL `Array`, or `None` if unsolved."
    if is_final_day(year, day) and part == "b": return Array("")
    p = puzzle_paths(year, day, token)[f"answer_{part}"]
    if is_file(p): return bstrip(read_text(p))
    _request_prose(year, day, token)
    return bstrip(read_text(p)) if is_file(p) else None


def submit_results(year, day, token):
    p = puzzle_paths(year, day, token).submit_results
    return read_json(p) if is_file(p) else json("[]")


def submit_answer(year, day, value, part=None, token=None, reopen=True, quiet=False):
    "Submit an answer to adventofcode.com (auto-detecting part a/b if `part` is None) with local guardrails and auto-retry."
    if _is_non_answer(value): raise AocdError(f"cowardly refusing to submit non-answer: {value!r}")
    tok = default_token() if token is None else bstr(token)
    v_arr = coerce(value)
    v_str = pystr(v_arr)
    if part is None: part = "a" if puzzle_answer(year, day, "a", tok) is None else "b"
    p_arr = _norm_part(bstr(part))
    p_str = pystr(p_arr)
    if p_str not in {"a", "b"}: raise AocdError('part must be "a" or "b"')

    v_res = vfi(v_arr)
    v_int = int(v_str) if (len(v_res[0]) == 1 and bool(v_res[0][0]) and len(r(v_arr, r"^-?\d+$").text)) else None
    skips = (Array("You gave an answer too recently"), Array("You don't seem to be solving the right level"))

    for res in aiter(submit_results(year, day, tok)):
        if not beq(res["part"], p_arr): continue
        msg = res["message"]
        if any(bool(bstartswith(sp, msg)) for sp in skips): continue
        pv = res["value"]
        is_right = bool(bstartswith(Array("That's the right answer"), msg))
        if is_right and not beq(v_arr, pv):
            if not quiet: print(f"Refusing {v_str!r}: part {p_str} already solved with {pystr(pv)!r}.")
            return None
        if bool(bhas("your answer is too high", msg)) and (v_int is None or v_int > int(pystr(pv))):
            if not quiet: print(f"Refusing {v_str!r}: {pystr(pv)!r} was already too high.")
            return None
        if bool(bhas("your answer is too low", msg)) and (v_int is None or v_int < int(pystr(pv))):
            if not quiet: print(f"Refusing {v_str!r}: {pystr(pv)!r} was already too low.")
            return None
        if beq(pv, v_arr):
            if not quiet: print(f"Refusing duplicate {v_str!r}: {pystr(msg)}")
            return None

    if p_str == "b" and beq(v_arr, puzzle_answer(year, day, "a", tok)):
        raise AocdError(f"cowardly refusing to submit {v_str} for part b, because that was the answer for part a")
    if (prev := puzzle_answer(year, day, p_str, tok)) is not None and len(prev):
        if not quiet: print(f"Part {p_str} already solved with {pystr(prev)!r}")
        return None

    url = catenate(puzzle_url(year, day), Array("/answer"))
    resp = http_post(url, token=tok, fields=Array(level="1" if p_str == "a" else "2", answer=v_arr))
    if int(resp.status) != 200: raise AocdError(f"HTTP {int(resp.status)} at {pystr(url)}")
    if bool(bhas("[Log In]", resp.body)): raise DeadTokenError(f"the auth token ...{pystr(take(-4, tok))} is dead")
    arts = tag_texts(resp.body, "article", strip_ws=True)
    if not arts: raise AocdError("Could not parse submission response <article>")
    message = arts[0]

    p_log = puzzle_paths(year, day, tok).submit_results
    hist = read_json(p_log) if is_file(p_log) else json("[]")
    write_json(p_log, catenate(hist, enclose(Array(part=p_arr, value=v_arr, when=_now_iso(0), message=message))))

    if bool(bhas("That's the right answer", message)):
        if reopen: webbrowser.open(pystr(catenate(puzzle_url(year, day), Array("#part2"))))
        if not (is_final_day(year, day) and p_str == "b"): _save_answer(year, day, v_arr, p_str, tok)
    elif bool(bhas("You gave an answer too recently", message)):
        wgrps = _wait_groups(message)
        if len(wgrps):
            m_a, s_a = wgrps[0][0], wgrps[0][1]
            delay(int(vfi(s_a)[1][0]) + (60 * int(vfi(m_a)[1][0]) if len(m_a) else 0))
            return submit_answer(year, day, v_arr, p_str, tok, reopen=reopen, quiet=quiet)
    if not quiet: print(pystr(message))
    return resp

## `Puzzle`

In [ ]:
#| export
class Puzzle:
    "Thin lazy-property facade for an Advent of Code puzzle (`input_data`, `examples`, `title`, `submit`)."
    def __init__(self, year, day, token=None):
        self.year, self.day = int(year), int(day)
        self.token = default_token() if token is None else bstr(token)
        self.url = puzzle_url(self.year, self.day)

    @cached_property
    def paths(self): return puzzle_paths(self.year, self.day, self.token)
    @cached_property
    def raw_input_data(self): return raw_input_data(self.year, self.day, self.token)
    @cached_property
    def input_data(self): return bsplit_lines(self.raw_input_data)
    @cached_property
    def title(self): return puzzle_title(self.year, self.day, self.token)
    @property
    def examples(self): return puzzle_examples(self.year, self.day, self.token)
    @property
    def submit_results(self): return submit_results(self.year, self.day, self.token)
    @property
    def answer_a(self):
        if (a := puzzle_answer(self.year, self.day, "a", self.token)) is None: raise AttributeError("answer_a")
        return a
    @property
    def answer_b(self):
        if (b := puzzle_answer(self.year, self.day, "b", self.token)) is None: raise AttributeError("answer_b")
        return b
    @property
    def answers(self): return self.answer_a, self.answer_b
    @property
    def answered_a(self): return bool(puzzle_answer(self.year, self.day, "a", self.token) is not None)
    @property
    def answered_b(self): return bool(puzzle_answer(self.year, self.day, "b", self.token) is not None)

    def submit(self, answer, part=None, reopen=True, quiet=False):
        "Submit `answer` for this puzzle (auto-detecting part a/b when `part` is None)."
        return submit_answer(self.year, self.day, answer, part=part, token=self.token, reopen=reopen, quiet=quiet)

    def __repr__(self): return f"<Puzzle({self.year}, {self.day})>"

### Tests

In [ ]:
_orig = (AOCD_DATA_DIR, AOCD_CONFIG_DIR, _token2id, http_get, http_post)
tmp_root = mkdir(host("temp"), unique=1, prefix="baocd_puz_")
try:
    AOCD_DATA_DIR = AOCD_CONFIG_DIR = tmp_root
    _token2id = None
    _prose = '<article><h2>--- Day 1: Secret Entrance ---</h2><pre><code>L68\nR48\n</code></pre><p>Ans <code>3</code></p></article>'
    _routes = {
        "https://adventofcode.com/settings": (200, '<code>ownerproof-42-1-a</code><span>Link to https://github.com/bpl</span>'),
        "https://adventofcode.com/2025/day/1/input": (200, "L68\nR48\n"),
        "https://adventofcode.com/2025/day/1": (200, _prose),
    }
    http_get = lambda url, token=None: Array(status=_routes[pystr(url)][0], body=Array(_routes[pystr(url)][1]))
    http_post = lambda url, token, fields: Array(status=200, body=Array("<article><p>That's the right answer!</p></article>"))

    p = Puzzle(2025, 1, token="live_tok_9999")
    test_beq(p.input_data, Array(["L68", "R48"]))
    test_beq(p.title, Array("Secret Entrance"))
    test_beq(p.examples[0].input_data, Array(["L68", "R48"]))
    p.submit(Array(10), reopen=False, quiet=True)
    test_beq(p.answer_a, Array("10"))
finally:
    AOCD_DATA_DIR, AOCD_CONFIG_DIR, _token2id, http_get, http_post = _orig
    remove(tmp_root, recurse=1)